# SafeX Solutions — Invoice Automation Demo

This notebook walks through the full pipeline end to end using a handful of sample orders from `data/sample_orders.csv`:

1. **Parse** — raw order text -> structured JSON (Azure OpenAI)
2. **Validate** — clean the parsed order and fill in missing prices (Pandas)
3. **Generate** — build a PDF invoice (ReportLab)
4. **Notify** — email the invoice to the customer (skipped/mocked here — see note below)

> **Note:** This notebook does **not** send real emails. The final step is mocked so the demo can be re-run freely without spamming an inbox or requiring real SMTP credentials. Run the actual `/generate-invoice` API endpoint (via `app.py`) to send a real email.

In [ ]:
import sys
import os
import json

# Allow importing the project modules (parser, validator, invoice_generator, notifier)
# from the parent directory of this notebook.
sys.path.append(os.path.abspath(".."))

import pandas as pd

from parser import parse_order
from validator import validate_order
from invoice_generator import generate_invoice

## Load sample orders

In [ ]:
orders_df = pd.read_csv("../data/sample_orders.csv")
orders_df.head(15)

In [ ]:
# Pick 4 sample orders to walk through: a couple of clean ones, one with a
# missing quantity/price, and one that's intentionally ambiguous.
sample_indices = [0, 4, 5, 8]
sample_orders = orders_df.loc[sample_indices, "order_text"].tolist()

for i, text in enumerate(sample_orders):
    print(f"[{i}] {text}\n")

## Step 1: Parse each order with the LLM

Requires valid Azure OpenAI credentials in `.env`. Each result is printed as formatted JSON.

In [ ]:
parsed_orders = []

for i, text in enumerate(sample_orders):
    parsed = parse_order(text)
    parsed_orders.append(parsed)
    print(f"--- Parsed order [{i}] ---")
    print(json.dumps(parsed, indent=2))
    print()

## Step 2: Validate + clean each parsed order

Fills in missing unit prices from the hardcoded product lookup, defaults invalid quantities, and surfaces warnings. Flagged orders are skipped here since they need human review, not validation.

In [ ]:
validated_orders = []

for i, parsed in enumerate(parsed_orders):
    print(f"--- Order [{i}] ---")
    if parsed.get("flagged"):
        print(f"FLAGGED — not validated. Reason: {parsed.get('flag_reason')}")
        validated_orders.append(None)
        print()
        continue

    validated = validate_order(parsed)
    validated_orders.append(validated)
    print(json.dumps(validated, indent=2))
    print()

## Step 3: Generate a PDF invoice for each validated order

In [ ]:
invoice_paths = []

for i, validated in enumerate(validated_orders):
    if validated is None:
        print(f"[{i}] Skipped (flagged order, no invoice generated).")
        invoice_paths.append(None)
        continue

    # Run from notebooks/, so invoices land in ../invoices/
    original_cwd = os.getcwd()
    os.chdir("..")
    try:
        path = generate_invoice(validated)
    finally:
        os.chdir(original_cwd)

    invoice_paths.append(path)
    print(f"[{i}] Invoice generated: {path}")

## Step 4: Email the invoice to the customer (MOCKED — no real send)

The real `send_invoice_email` function in `notifier.py` sends a live email via Gmail SMTP.
We intentionally **do not call it here** so this notebook can be re-run safely without
sending real emails or requiring a working Gmail app password to just view the demo.
Instead we print what *would* be sent. Use the FastAPI endpoint for an actual send.

In [ ]:
from datetime import datetime, timezone

def mock_send_invoice_email(to_email, customer_name, pdf_path):
    """Stand-in for notifier.send_invoice_email — does not touch SMTP at all."""
    return {
        "status": "mocked",
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "to_email": to_email,
        "customer_name": customer_name,
        "pdf_path": pdf_path,
        "note": "Real email NOT sent — mocked for notebook demo purposes.",
    }

for i, (validated, path) in enumerate(zip(validated_orders, invoice_paths)):
    if validated is None or path is None:
        print(f"[{i}] Skipped (flagged order, no email to send).")
        continue

    result = mock_send_invoice_email(
        to_email="demo.customer@example.com",
        customer_name=validated.get("customer_name", "Customer"),
        pdf_path=path,
    )
    print(f"--- Order [{i}] email result (mocked) ---")
    print(json.dumps(result, indent=2))
    print()

## Summary

This walkthrough mirrors exactly what the `/generate-invoice` FastAPI endpoint does internally:
`parse_order` → `validate_order` → `generate_invoice` → `send_invoice_email`. To run the real thing
end to end (including a live email send), start the server with `uvicorn app:app --reload` and POST
to `/generate-invoice` — see the README for a sample `curl` command.